# Object Detection

In [ ]:
import torch
print("torch is installed:")
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("CUDA device count:", torch.cuda.device_count())
else:
    print("No CUDA GPU detected")
print("torch version: " + torch.__version__)
print("torch-cuda version: " + torch.version.cuda)

In [ ]:
from mmdet3d.apis import init_model, inference_detector

CONFIG = "../_weights_and_configs/configs/votenet/votenet_8xb8_scannet-3d.py"
CHECKPOINT = "../_weights_and_configs/votenet_8x8_scannet-3d-18class_20210823_234503-cf8134fa.pth"
model = init_model(CONFIG, CHECKPOINT,device='cuda:0')
inference_detector(model, '../_input/scene0000_00.bin')

In [ ]:
import numpy as np
import torch
from mmengine.dataset import pseudo_collate
from mmdet3d.apis import init_model
from mmdet3d.structures import LiDARInstance3DBoxes

CONFIG = "../_weights_and_configs/configs/votenet/votenet_8xb8_scannet-3d.py"
CHECKPOINT = "../_weights_and_configs/votenet_8x8_scannet-3d-18class_20210823_234503-cf8134fa.pth"

# Load your ASCII or .bin point cloud
points = np.fromfile('../_input/000008.bin', dtype=np.float32).reshape(-1,4)
xyz = points[:, :3]
rgb = np.zeros_like(xyz)         # dummy RGB if none
pc_input = np.concatenate([xyz, rgb], axis=1).astype(np.float32)

# Wrap in a dict that matches the ScanNet dataset format
data = dict(
    points=torch.from_numpy(pc_input),
    pts_filename='',                    # required key
    box_type_3d=LiDARInstance3DBoxes   # for VoteNet
)

# Collate into a batch
batch = pseudo_collate([data])

# Init model
model = init_model(CONFIG, CHECKPOINT, device='cuda:0')

# Inference
with torch.no_grad():
    result = model.test_step(batch)

# Extract results
boxes_3d = result[0]['boxes_3d']
scores_3d = result[0]['scores_3d']
labels_3d = result[0]['labels_3d']

print("Detected boxes:", len(boxes_3d))

In [ ]:
PcdPath = "/home/jelle-vermandere/Documents/Github/DRM/_input/Office1 - Cloud.txt"

In [ ]:
import numpy as np
import tempfile
from mmdet3d.apis import init_model, inference_detector

# -----------------------------
# SETTINGS
# -----------------------------
CONFIG = "../_weights_and_configs/configs/votenet/votenet_8xb8_scannet-3d.py"
CHECKPOINT = "../_weights_and_configs/votenet_8x8_scannet-3d-18class_20210823_234503-cf8134fa.pth"
ASCII_FILE = "../_input/Office1 - Cloud.txt"


In [ ]:

# -----------------------------
# LOAD ASCII POINT CLOUD
# -----------------------------
points = np.loadtxt(ASCII_FILE)

xyz = points[:, 0:3]
rgb = points[:, 6:9] / 255.0   # normalize colors (important)

# VoteNet expects XYZRGB
pc = np.concatenate([xyz, rgb], axis=1)

# -----------------------------
# LOAD MODEL
# -----------------------------
model = init_model(CONFIG, CHECKPOINT, device='cuda:0')


In [ ]:
print(pc[:100].shape)

In [ ]:
import numpy as np
import torch
from mmengine.dataset import pseudo_collate
from mmdet3d.apis import init_model
# prepare input
points = pc[:100].astype(np.float32)
pointsDict = dict(points=torch.from_numpy(points))
# collate (simulate batch dimension)
collated = pseudo_collate([pointsDict])
# inference
with torch.no_grad():
    result = model.test_step(collated)
# -----------------------------
# RUN INFERENCE
# -----------------------------
#result, data = inference_detector(model, collated)

print(result)

In [ ]:
import open3d as o3d
import numpy as np
import json

# Example data
with open("/home/jelle-vermandere/Documents/Github/DRM/_output/scene0000_00.json") as f:
    data = json.load(f)
    print(data)


# Parameters
score_threshold = 0.8  # Only show boxes with score > threshold

# Colormap for labels
label_colors = [
    [1, 0, 0],  # red
    [0, 1, 0],  # green
    [0, 0, 1],  # blue
    [1, 1, 0],  # yellow
    [1, 0, 1],  # magenta
    [0, 1, 1],  # cyan
]

def create_bbox(center, size, rotation_z=0.0, color=[1,0,0]):
    """
    Create an Open3D OrientedBoundingBox from center, size, rotation, and color
    """
    bbox = o3d.geometry.OrientedBoundingBox()
    bbox.center = center
    bbox.extent = size
    R = o3d.geometry.get_rotation_matrix_from_axis_angle([0, 0, rotation_z])
    bbox.R = R
    bbox.color = color
    return bbox

# Create Open3D geometries
geometries = []

for label, score, box in zip(data["labels_3d"], data["scores_3d"], data["bboxes_3d"]):
    if score < score_threshold:
        continue

    # box = [x, y, z, dx, dy, dz, rotation_z]
    center = np.array(box[:3])
    size = np.array(box[3:6])
    rotation_z = box[6]
    color = label_colors[label % len(label_colors)]
    bbox = create_bbox(center, size, rotation_z, color)
    geometries.append(bbox)

# Add coordinate frame
geometries.append(o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0))


In [ ]:

# Visualize
o3d.visualization.draw_geometries(geometries)

In [ ]:
import trimesh
import numpy as np

# Example data
# Example data
with open("/home/jelle-vermandere/Documents/Github/DRM/_output/000017.json") as f:
    data = json.load(f)
    print(data)

# Parameters
score_threshold = 0.5  # Only show boxes with score > threshold

# Colormap for labels
label_colors = [
    [1, 0, 0, 0.5],  # red, alpha 0.5
    [0, 1, 0, 0.5],  # green
    [0, 0, 1, 0.5],  # blue
    [1, 1, 0, 0.5],  # yellow
    [1, 0, 1, 0.5],  # magenta
    [0, 1, 1, 0.5],  # cyan
]

def create_trimesh_box(center, size, rotation_z=0.0, color=[1,0,0,0.5]):
    """
    Create a trimesh Box mesh with given center, size, rotation, and color.
    """
    # Box is created centered at origin
    box = trimesh.creation.box(extents=size, transform=None)
    
    # Rotation matrix around z-axis
    c, s = np.cos(rotation_z), np.sin(rotation_z)
    R = np.array([
        [c, -s, 0, 0],
        [s,  c, 0, 0],
        [0,  0, 1, 0],
        [0,  0, 0, 1]
    ])
    
    # Translation to center
    T = np.eye(4)
    T[:3, 3] = center

    # Apply transform
    box.apply_transform(T @ R)
    
    # Set color (RGBA)
    box.visual.face_colors = color
    
    return box

# Create list of meshes
meshes = []

for label, score, box in zip(data["labels_3d"], data["scores_3d"], data["bboxes_3d"]):
    if score < score_threshold:
        continue

    center = np.array(box[:3])
    size = np.array(box[3:6])
    rotation_z = box[6]
    color = label_colors[label % len(label_colors)]
    
    mesh = create_trimesh_box(center, size, rotation_z, color)
    meshes.append(mesh)



In [ ]:
def load_bin_pointcloud(file_path):
    """Load KITTI-style .bin point cloud"""
    points = np.fromfile(file_path, dtype=np.float32).reshape(-1, 6)  # x, y, z, rgb
    # Normalize reflectance to [0,255] for colors
    reflectance = points[:, 3]
    colors = np.stack([reflectance, reflectance, reflectance], axis=1)  # grayscale
    colors = (colors / colors.max() * 255).astype(np.uint8)
    cloud = trimesh.points.PointCloud(points[:, :3], colors=points[:, 3:6]/255)
    return cloud

# -----------------------------
# Load point cloud
# -----------------------------
pointcloud_file = "/home/jelle-vermandere/Documents/Github/DRM/_input/000017.bin"  # Replace with your path
pcd = load_bin_pointcloud(pointcloud_file)
meshes.append(pcd)

In [ ]:
# Combine meshes for visualization
scene = trimesh.Scene(meshes)

# Show interactive visualization
scene.show()